In [1]:
import numpy as np
import cvxpy as cp
from scipy import optimize

In [7]:
means = np.array([0.1, -0.05, 0.15])
mean_vols = np.array([0.04, 0.02, 0.1])
vols = np.array([0.2, 0.2, 0.15])
cor_mat = np.array([[1, 0.5, 0.2], [0.5, 1, 0.6], [0.2, 0.6, 1.0]])
cov_mat = np.diag(vols) @ cor_mat @ np.diag(vols).T
b = np.ones(len(means)) / len(means)

cov_mat

array([[0.04  , 0.02  , 0.006 ],
       [0.02  , 0.04  , 0.018 ],
       [0.006 , 0.018 , 0.0225]])

In [9]:
w = cp.Variable(len(means))
gamma = cp.Parameter(nonneg=True)
active_return = (w - b) @ means
tracking_error_var = cp.quad_form((w - b), cov_mat)
prob = cp.Problem(
    cp.Minimize(tracking_error_var - gamma * active_return),
    constraints=[cp.sum(w) == 1],
)

In [ ]:
def bisection(
    target: float, gamma_range: list[float, float], is_risk_target: bool = True
):
    def eval(tmp_gamma):
        nonlocal target
        gamma.value = tmp_gamma
        prob.solve()
        if is_risk_target:
            return np.sqrt(tracking_error_var.value) - target
        else:
            return active_return.value - target

    return optimize.bisect(eval, gamma_range[0], gamma_range[1])

In [14]:
for target in [0.01, 0.02, 0.03, 0.04, 0.05]:
    optimal_gamma = bisection(
        target=target, gamma_range=[0.01, 100], is_risk_target=True
    )
    gamma.value = optimal_gamma
    prob.solve()
    print("Tracking error vol target:", target)
    print(
        "tracking error vol:",
        np.sqrt(tracking_error_var.value),
        "active return:",
        active_return.value,
    )
    print("Optimal weights:", w.value)
    print("----------------------------------")

Tracking error vol target: 0.01
tracking error vol: 0.00999999999921385 active return: 0.013126852516922055
Optimal weights: [0.35149868 0.26315773 0.38534359]
----------------------------------
Tracking error vol target: 0.02
tracking error vol: 0.01999999999957529 active return: 0.026253705035350124
Optimal weights: [0.36966402 0.19298214 0.43735384]
----------------------------------
Tracking error vol target: 0.03
tracking error vol: 0.030000000000105803 active return: 0.03938055755400066
Optimal weights: [0.38782937 0.12280654 0.48936409]
----------------------------------
Tracking error vol target: 0.04
tracking error vol: 0.04000000000045892 active return: 0.05250741007241809
Optimal weights: [0.40599472 0.05263094 0.54137435]
----------------------------------
Tracking error vol target: 0.05
tracking error vol: 0.050000000000986984 active return: 0.06563426259106529
Optimal weights: [ 0.42416006 -0.01754466  0.5933846 ]
----------------------------------
